# Scratch Vision Transformer

## Objective

This notebooks trains a Sctrach Vision Transformer model.
Training data is sourced from Data_generator notebooks that we have saved earlier

4 notebooks are used for training 
1 notebooks is used for validation 

Total number of samples used for training  are: `101771`

Total number of samples used for validation are: `25452`


# WandB setup

In [ ]:
!pip install --upgrade wandb -q

In [ ]:
import wandb
wandb.login()

# Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
from IPython.display import Audio, display

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchaudio
import torch.optim as optim 
import torchaudio.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

from glob import glob
import random
from tqdm import tqdm
import soundfile as sf

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import os
import sys
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks.early_stopping import EarlyStopping

from sklearn.metrics import f1_score , accuracy_score, classification_report, confusion_matrix
import math

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("github_access")

!rm -rf /kaggle/working/dl-genai-project-26-t1

!git clone https://{secret_value_0}@github.com/Aryanch9797/dl-genai-project-26-t1.git

repo_path = "/kaggle/working/dl-genai-project-26-t1"
if repo_path not in sys.path:
    sys.path.append(repo_path)

from src.Trainers.custom_trainer import training_model
from src.dataset.test_dataset import test_mashed_dataset
from src.dataset.train_val_dataset import MelSpectrogramDataset
from src.models.scratch_vision_transformer import scratch_vision_transformer
from src.inference import prediction

# Data Loader

In [ ]:
val_path ="/kaggle/input/notebooks/aryanchauhan97971234/data-generator-ipynb/train_data/"
train_paths = [
    "/kaggle/input/notebooks/aryanchauhan97971234/data-generator-2-ipynb/train_data/",
    "/kaggle/input/notebooks/aryanchauhan97971234/data-generator-3-ipynb/train_data/",
    "/kaggle/input/notebooks/aryanchauhan97971234/data-generator-4-ipynb/train_data/",
    "/kaggle/input/notebooks/aryanchauhan97971234/data-generator-5-ipynb/train_data/"
]
test_csv = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv" , index_col='id')
sample_submission = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv" , index_col='id' )
path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/"

In [ ]:
train_dataset = MelSpectrogramDataset(False, train_paths, val_path)
val_dataset = MelSpectrogramDataset(True, train_paths, val_path)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4,pin_memory=True,persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4,pin_memory=True,persistent_workers=True)

test_data = test_mashed_dataset(path, test_csv, 16000, 10.24)
# batch_size 1 for test data so all audio chunks of one sample can be processed together
test_loader = DataLoader(test_data , batch_size=1, shuffle=False, num_workers=4,pin_memory=True, persistent_workers=True)

In [ ]:
label_to_genre = {
    0: "blues", 1: "classical", 2: "country", 3: "disco", 4: "hiphop",
    5: "jazz", 6: "metal", 7: "pop", 8: "reggae", 9: "rock"
}

# ViT model

In [ ]:
model = scratch_vision_transformer(7,0.0003,0.001,5,2,0.5,40,0.3)

# Model traning

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor='val/f1_macro',    
    dirpath='saved_models/', 
    filename='vit-best-{epoch:02d}-{val_loss:.2f}',
    save_top_k=1,            
    mode='max'            
)

early_stop_callback = EarlyStopping(
    monitor='val/f1_macro', 
    min_delta=0.00,         
    patience=5,             
    verbose=True,
    mode='max'              
)

wandb_logger = WandbLogger(project="Audio_Mashup", name="vit_7_layers_v4_dr_0.3")
wandb_logger.experiment.define_metric("epoch")
wandb_logger.experiment.define_metric("train/*", step_metric="epoch")
wandb_logger.experiment.define_metric("val/*", step_metric="epoch")


trainer = pl.Trainer(
    max_epochs=40, 
    accelerator="gpu", 
    devices=1,  
    logger=wandb_logger,
    callbacks=[checkpoint_callback, early_stop_callback] 
)


trainer.fit(model, train_loader, val_loader)

best_ckpt_path = checkpoint_callback.best_model_path
checkpoint = torch.load(best_ckpt_path)
best_state_dict = checkpoint['state_dict']

In [ ]:
import kagglehub

MODEL_UPLOAD_DIR = "/kaggle/working/Scratch_ViT" 
os.makedirs(MODEL_UPLOAD_DIR, exist_ok=True)

MODEL_SAVE_PATH = os.path.join(MODEL_UPLOAD_DIR, "Scratch_ViT.pth")
torch.save(best_state_dict, MODEL_SAVE_PATH)     # model_trained
print(f"Model saved to {MODEL_SAVE_PATH}")

KAGGLE_USERNAME = 'aryanchauhan97971234' 
MODEL = 'Scratch_ViT_7_layers_v1'
FRAMEWORK = 'pytorch'
VARIATION = '0.3drop_v1'

handle = f'{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}'

print(f"Uploading model from {MODEL_UPLOAD_DIR} to {handle}...")

kagglehub.model_upload(
    handle,                     
    MODEL_UPLOAD_DIR,           
    license_name="Apache 2.0", 
    version_notes="4th run with transfer, lr scheduler"
)
print("Upload complete!")